# Notebook 49: SOPR Variants Entry Signal Comparison

**Objective**: Test all SOPR variants as entry signals to find optimal capitulation detection.

## SOPR Variants Available:
1. **SOPR** - Standard Spent Output Profit Ratio
2. **STH-SOPR** - Short-Term Holder SOPR (< 155 days)
3. **LTH-SOPR** - Long-Term Holder SOPR (> 155 days)
4. **Entity-Adjusted SOPR** - Filters internal transfers (exchanges, etc.)

## Entry Logic:
- SOPR < 1 = Coins moving at a loss = Capitulation = Entry opportunity
- Exit: 30% trailing stop (validated in previous research)

## Test Matrix:
- Each variant standalone
- Combinations (AND logic)
- Different thresholds (< 1.0, < 0.98, < 0.95)
- Cooldown periods

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../data/daily")

# Load all data
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
sopr_lth = pd.read_parquet(DATA_DIR / "sopr_lth.parquet").rename(columns={"value": "sopr_lth"}).set_index("time")

# Entity-adjusted SOPR (newly downloaded from Glassnode)
sopr_adj = pd.read_parquet(DATA_DIR / "sopr_adjusted.parquet")
# The index is already 'date' from how we saved it
sopr_adj.index.name = 'time'

# Fix timezone mismatch - Glassnode is tz-naive, Bitcoin Lab is tz-aware (UTC)
if sopr_adj.index.tz is None:
    sopr_adj.index = sopr_adj.index.tz_localize('UTC')

# Also load realized loss for STRAT-002 comparison
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

# Debug: check index types
print("Index types:")
print(f"  price: {price.index.dtype}, tz={getattr(price.index, 'tz', 'N/A')}")
print(f"  sopr_adj: {sopr_adj.index.dtype}, tz={getattr(sopr_adj.index, 'tz', 'N/A')}")

# Merge all
df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(sopr_lth, how='inner').join(sopr_adj, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

# Add realized loss z-score for STRAT-002
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()

# Filter to 2019+ (post-2018 crash, mature market)
df = df[df.index >= '2019-01-01'].dropna()

print(f"\nData loaded: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Years: {(df.index.max() - df.index.min()).days / 365.25:.1f}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSOPR variants summary:")
print(df[['sopr', 'sopr_sth', 'sopr_lth', 'sopr_adjusted']].describe().round(3))

In [ ]:
# Visualize SOPR variants correlation
print("SOPR VARIANT CORRELATIONS")
print("="*60)
corr = df[['sopr', 'sopr_sth', 'sopr_lth', 'sopr_adjusted']].corr()
print(corr.round(3))
print("\n(High correlation = redundant signals, low = complementary)")

In [ ]:
# Count days below 1 for each variant
print("DAYS BELOW 1.0 (Capitulation Days)")
print("="*60)
for col in ['sopr', 'sopr_sth', 'sopr_lth', 'sopr_adjusted']:
    below_1 = (df[col] < 1).sum()
    pct = below_1 / len(df) * 100
    print(f"{col:<20}: {below_1:>5} days ({pct:>5.1f}%)")

print("\nDAYS BELOW 0.98 (Deeper Capitulation)")
print("="*60)
for col in ['sopr', 'sopr_sth', 'sopr_lth', 'sopr_adjusted']:
    below = (df[col] < 0.98).sum()
    pct = below / len(df) * 100
    print(f"{col:<20}: {below:>5} days ({pct:>5.1f}%)")

In [ ]:
def run_backtest(data, entries, trail=0.30, init_cash=100000):
    """Run backtest with trailing stop exit."""
    if entries.sum() == 0:
        return None
    return vbt.Portfolio.from_signals(
        close=data['price'],
        entries=entries,
        sl_stop=trail,
        sl_trail=True,
        stop_exit_price='close',
        fees=0.001,
        init_cash=init_cash,
        freq='D'
    )

def get_metrics(pf, years):
    """Extract key metrics from portfolio."""
    if pf is None or pf.trades.count() == 0:
        return None
    return {
        'return': pf.total_return() * 100,
        'cagr': pf.annualized_return() * 100,
        'sharpe': pf.sharpe_ratio(),
        'sortino': pf.sortino_ratio(),
        'max_dd': pf.max_drawdown() * 100,
        'win_rate': pf.trades.win_rate() * 100,
        'trades': pf.trades.count(),
        'trades_yr': pf.trades.count() / years,
        'profit_factor': pf.trades.profit_factor(),
        'final_value': pf.final_value()
    }

years = (df.index.max() - df.index.min()).days / 365.25

## Part 1: Standalone Entry Signals (< 1.0)

In [ ]:
# Define entry conditions for each variant (standard < 1 threshold)
entry_signals = {
    'SOPR < 1': df['sopr'] < 1,
    'STH-SOPR < 1': df['sopr_sth'] < 1,
    'LTH-SOPR < 1': df['sopr_lth'] < 1,
    'Entity-Adj < 1': df['sopr_adjusted'] < 1,
}

# Convert conditions to actual entry signals (only on first day of condition)
entries = {}
for name, cond in entry_signals.items():
    entries[name] = cond & ~cond.shift(1).fillna(False)

# Add STRAT-002 baseline for comparison
strat002_cond = (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['rl_zscore'] > 0.5)
entries['STRAT-002 (triple)'] = strat002_cond & ~strat002_cond.shift(1).fillna(False)

# Count entries
print("ENTRY SIGNAL FREQUENCY (< 1.0 threshold)")
print("="*60)
for name, entry in entries.items():
    print(f"{name:<20}: {entry.sum():>4} entries ({entry.sum()/years:.1f}/year)")

In [ ]:
# Run backtests for all variants
results = {}
for name, entry in entries.items():
    pf = run_backtest(df, entry, trail=0.30)
    metrics = get_metrics(pf, years)
    if metrics:
        results[name] = metrics

# Display comparison
print("\nSTANDALONE SOPR VARIANT COMPARISON (30% trailing stop)")
print("="*120)
print(f"{'Signal':<22} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'WinRate':>8} {'Trades':>8} {'/Year':>6} {'PF':>6}")
print("-"*120)

for name, m in sorted(results.items(), key=lambda x: x[1]['return'], reverse=True):
    print(f"{name:<22} {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['win_rate']:>7.0f}% {m['trades']:>8} {m['trades_yr']:>6.1f} {m['profit_factor']:>6.2f}")

bh = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100
print("-"*120)
print(f"{'Buy & Hold':<22} {bh:>+9.0f}%")

## Part 2: Combination Signals (AND logic)

In [ ]:
# Define combination entry conditions
combo_signals = {
    # Two-way combinations
    'SOPR + STH': (df['sopr'] < 1) & (df['sopr_sth'] < 1),
    'SOPR + Adj': (df['sopr'] < 1) & (df['sopr_adjusted'] < 1),
    'STH + Adj': (df['sopr_sth'] < 1) & (df['sopr_adjusted'] < 1),
    'SOPR + LTH': (df['sopr'] < 1) & (df['sopr_lth'] < 1),
    
    # Three-way combinations
    'SOPR + STH + Adj': (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['sopr_adjusted'] < 1),
    'SOPR + STH + LTH': (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['sopr_lth'] < 1),
    
    # All four
    'ALL < 1': (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['sopr_lth'] < 1) & (df['sopr_adjusted'] < 1),
}

# Convert to entry signals
combo_entries = {}
for name, cond in combo_signals.items():
    combo_entries[name] = cond & ~cond.shift(1).fillna(False)

# Count entries
print("COMBINATION SIGNAL FREQUENCY")
print("="*60)
for name, entry in combo_entries.items():
    print(f"{name:<22}: {entry.sum():>4} entries ({entry.sum()/years:.1f}/year)")

In [ ]:
# Run backtests for combinations
combo_results = {}
for name, entry in combo_entries.items():
    pf = run_backtest(df, entry, trail=0.30)
    metrics = get_metrics(pf, years)
    if metrics:
        combo_results[name] = metrics

# Display comparison
print("\nCOMBINATION SIGNALS COMPARISON (30% trailing stop)")
print("="*120)
print(f"{'Signal':<22} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'WinRate':>8} {'Trades':>8} {'/Year':>6} {'PF':>6}")
print("-"*120)

for name, m in sorted(combo_results.items(), key=lambda x: x[1]['return'], reverse=True):
    print(f"{name:<22} {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['win_rate']:>7.0f}% {m['trades']:>8} {m['trades_yr']:>6.1f} {m['profit_factor']:>6.2f}")

print("-"*120)
print(f"{'Buy & Hold':<22} {bh:>+9.0f}%")

## Part 3: Threshold Sensitivity Analysis

In [ ]:
# Test different thresholds for the best standalone signal
print("THRESHOLD SENSITIVITY - Entity-Adjusted SOPR")
print("="*100)
print(f"{'Threshold':<12} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'/Year':>8} {'WinRate':>8}")
print("-"*100)

for thresh in [1.02, 1.00, 0.99, 0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92]:
    cond = df['sopr_adjusted'] < thresh
    entry = cond & ~cond.shift(1).fillna(False)
    pf = run_backtest(df, entry, trail=0.30)
    m = get_metrics(pf, years)
    if m:
        print(f"< {thresh:<10} {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_yr']:>8.1f} {m['win_rate']:>7.0f}%")

In [ ]:
# Test different thresholds for STH-SOPR
print("\nTHRESHOLD SENSITIVITY - STH-SOPR")
print("="*100)
print(f"{'Threshold':<12} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'/Year':>8} {'WinRate':>8}")
print("-"*100)

for thresh in [1.02, 1.00, 0.99, 0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92]:
    cond = df['sopr_sth'] < thresh
    entry = cond & ~cond.shift(1).fillna(False)
    pf = run_backtest(df, entry, trail=0.30)
    m = get_metrics(pf, years)
    if m:
        print(f"< {thresh:<10} {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_yr']:>8.1f} {m['win_rate']:>7.0f}%")

## Part 4: Trail Stop Optimization for Best Signals

In [ ]:
# Find best signal from above, test different trail stops
# Testing Entity-Adjusted SOPR < 1
best_cond = df['sopr_adjusted'] < 1
best_entry = best_cond & ~best_cond.shift(1).fillna(False)

print("TRAIL STOP SENSITIVITY - Entity-Adjusted SOPR < 1")
print("="*100)
print(f"{'Trail %':<10} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'/Year':>8} {'WinRate':>8}")
print("-"*100)

for trail in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    pf = run_backtest(df, best_entry, trail=trail)
    m = get_metrics(pf, years)
    if m:
        print(f"{trail*100:>8.0f}% {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_yr']:>8.1f} {m['win_rate']:>7.0f}%")

In [ ]:
# Test STH-SOPR with different trails
sth_cond = df['sopr_sth'] < 1
sth_entry = sth_cond & ~sth_cond.shift(1).fillna(False)

print("\nTRAIL STOP SENSITIVITY - STH-SOPR < 1")
print("="*100)
print(f"{'Trail %':<10} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'/Year':>8} {'WinRate':>8}")
print("-"*100)

for trail in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    pf = run_backtest(df, sth_entry, trail=trail)
    m = get_metrics(pf, years)
    if m:
        print(f"{trail*100:>8.0f}% {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_yr']:>8.1f} {m['win_rate']:>7.0f}%")

## Part 5: Entity-Adjusted + STH Combination Deep Dive

In [ ]:
# Test Entity-Adjusted + STH combination with various thresholds
print("ENTITY-ADJUSTED + STH-SOPR COMBINATION")
print("="*120)
print(f"{'Adj Thresh':<12} {'STH Thresh':<12} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'WinRate':>8}")
print("-"*120)

for adj_thresh in [1.00, 0.98, 0.96]:
    for sth_thresh in [1.00, 0.98, 0.96]:
        cond = (df['sopr_adjusted'] < adj_thresh) & (df['sopr_sth'] < sth_thresh)
        entry = cond & ~cond.shift(1).fillna(False)
        pf = run_backtest(df, entry, trail=0.30)
        m = get_metrics(pf, years)
        if m:
            print(f"< {adj_thresh:<10} < {sth_thresh:<10} {m['return']:>+9.0f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['win_rate']:>7.0f}%")

## Part 6: Trade-by-Trade Analysis of Best Signal

In [ ]:
# Run best signal and show trade log
# Using Entity-Adjusted SOPR < 1 with 30% trail (or adjust based on results above)
best_cond = df['sopr_adjusted'] < 1
best_entry = best_cond & ~best_cond.shift(1).fillna(False)
pf_best = run_backtest(df, best_entry, trail=0.30)

print("ENTITY-ADJUSTED SOPR < 1 TRADE LOG (30% trail)")
print("="*120)
trades = pf_best.trades.records_readable
print(trades.to_string())

In [ ]:
# Compare with STRAT-002 trade log
strat002_cond = (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['rl_zscore'] > 0.5)
strat002_entry = strat002_cond & ~strat002_cond.shift(1).fillna(False)
pf_strat002 = run_backtest(df, strat002_entry, trail=0.30)

print("\nSTRAT-002 (SOPR + STH + RL) TRADE LOG (30% trail)")
print("="*120)
if pf_strat002:
    trades_s2 = pf_strat002.trades.records_readable
    print(trades_s2.to_string())

## Part 7: Visualize Entry Points

In [ ]:
import matplotlib.pyplot as plt

# Plot price with entry signals
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Price with entries
ax1 = axes[0]
ax1.semilogy(df.index, df['price'], 'b-', alpha=0.7, label='BTC Price')

# Mark entries
adj_entries = df['sopr_adjusted'] < 1
adj_entry_points = adj_entries & ~adj_entries.shift(1).fillna(False)
ax1.scatter(df.index[adj_entry_points], df['price'][adj_entry_points], 
            c='green', s=100, marker='^', label='Entity-Adj < 1 Entry', zorder=5)

ax1.set_ylabel('Price (log)')
ax1.legend(loc='upper left')
ax1.set_title('BTC Price with Entity-Adjusted SOPR Entry Signals')
ax1.grid(True, alpha=0.3)

# Entity-Adjusted SOPR
ax2 = axes[1]
ax2.plot(df.index, df['sopr_adjusted'], 'purple', alpha=0.7, label='Entity-Adj SOPR')
ax2.axhline(y=1, color='red', linestyle='--', alpha=0.5)
ax2.fill_between(df.index, df['sopr_adjusted'], 1, where=(df['sopr_adjusted'] < 1), 
                 color='green', alpha=0.3, label='< 1 (Capitulation)')
ax2.set_ylabel('SOPR')
ax2.legend(loc='upper right')
ax2.set_ylim(0.8, 1.3)
ax2.grid(True, alpha=0.3)

# STH-SOPR for comparison
ax3 = axes[2]
ax3.plot(df.index, df['sopr_sth'], 'orange', alpha=0.7, label='STH-SOPR')
ax3.axhline(y=1, color='red', linestyle='--', alpha=0.5)
ax3.fill_between(df.index, df['sopr_sth'], 1, where=(df['sopr_sth'] < 1), 
                 color='green', alpha=0.3, label='< 1 (Capitulation)')
ax3.set_ylabel('STH-SOPR')
ax3.legend(loc='upper right')
ax3.set_ylim(0.8, 1.3)
ax3.grid(True, alpha=0.3)
ax3.set_xlabel('Date')

plt.tight_layout()
plt.savefig('sopr_variants_entries.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sopr_variants_entries.png")

## Summary & Conclusions

In [ ]:
# Final summary table - all signals ranked by return
all_results = {**results, **combo_results}

print("\n" + "="*120)
print("FINAL RANKING - ALL SOPR ENTRY SIGNALS (30% trailing stop)")
print("="*120)
print(f"{'Rank':<5} {'Signal':<25} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'WinRate':>8}")
print("-"*120)

sorted_results = sorted(all_results.items(), key=lambda x: x[1]['return'], reverse=True)
for i, (name, m) in enumerate(sorted_results, 1):
    print(f"{i:<5} {name:<25} {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['win_rate']:>7.0f}%")

print("-"*120)
print(f"{'':>5} {'Buy & Hold':<25} {bh:>+9.0f}%")
print("\n" + "="*120)
print("KEY FINDINGS:")
print("="*120)
print("1. [FILL BASED ON RESULTS]")
print("2. [FILL BASED ON RESULTS]")
print("3. [FILL BASED ON RESULTS]")